In [1]:
from google.cloud import bigquery
import os
import re

client = bigquery.Client()

PROJECT_ID = client.project          # uses the notebook's active project
BQ_LOCATION = "US"                   # adjust only if your lab requires e.g. "us-central1"
DATASET_ID = "fraud_lab"
RAW_TABLE = "fraud_data_raw"
TRAIN_TABLE = "fraud_training_data"

raw_table_id = f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}"
train_table_id = f"{PROJECT_ID}.{DATASET_ID}.{TRAIN_TABLE}"

print("PROJECT_ID:", PROJECT_ID)
print("raw_table_id:", raw_table_id)
print("train_table_id:", train_table_id)

PROJECT_ID: qwiklabs-gcp-02-5c5074b12ae4
raw_table_id: qwiklabs-gcp-02-5c5074b12ae4.fraud_lab.fraud_data_raw
train_table_id: qwiklabs-gcp-02-5c5074b12ae4.fraud_lab.fraud_training_data


In [5]:
dataset = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset.location = "US"  # change only if your lab region is different

client.create_dataset(dataset, exists_ok=True)
print("Dataset ready:", f"{PROJECT_ID}.{DATASET_ID}")

Dataset ready: qwiklabs-gcp-02-5c5074b12ae4.fraud_lab


In [7]:
gcs_uri = "gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)

load_job = client.load_table_from_uri(
    gcs_uri,
    raw_table_id,
    job_config=job_config,
    location="US",
)
load_job.result()

table = client.get_table(raw_table_id)
print("Loaded rows:", table.num_rows)
print("Schema:", [f"{f.name}:{f.field_type}" for f in table.schema])

Loaded rows: 50000
Schema: ['Applicant_ID:INTEGER', 'Age:INTEGER', 'Employment_Status:STRING', 'Income:INTEGER', 'Number_of_Dependents:INTEGER', 'Amount_Requested:INTEGER', 'Previous_Assistance_Received:BOOLEAN', 'Previous_Assistance_Date:DATE', 'Supporting_Doc_Verified:BOOLEAN', 'Application_Frequency_Last_Year:INTEGER', 'IP_Address:STRING', 'Device_Type:STRING', 'Application_Date:DATE', 'Fraudulent:INTEGER']


In [8]:
%load_ext bigquery_magics

In [16]:
%%bigquery
CREATE OR REPLACE EXTERNAL TABLE fraud_lab.fraud_data_raw_ext
OPTIONS (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv'],
  skip_leading_rows = 1
);

Query is running:   0%|          |

""


In [17]:
%%bigquery
CREATE OR REPLACE TABLE fraud_lab.fraud_data_raw AS
SELECT * FROM fraud_lab.fraud_data_raw_ext;

Query is running:   0%|          |

""


In [19]:
schema = client.get_table(raw_table_id).schema
[(f.name, f.field_type) for f in schema]

[('Applicant_ID', 'INTEGER'),
 ('Age', 'INTEGER'),
 ('Employment_Status', 'STRING'),
 ('Income', 'INTEGER'),
 ('Number_of_Dependents', 'INTEGER'),
 ('Amount_Requested', 'INTEGER'),
 ('Previous_Assistance_Received', 'BOOLEAN'),
 ('Previous_Assistance_Date', 'DATE'),
 ('Supporting_Doc_Verified', 'BOOLEAN'),
 ('Application_Frequency_Last_Year', 'INTEGER'),
 ('IP_Address', 'STRING'),
 ('Device_Type', 'STRING'),
 ('Application_Date', 'DATE'),
 ('Fraudulent', 'INTEGER')]

In [20]:
preview_sql = f"SELECT * FROM `{raw_table_id}` LIMIT 10"
df_preview = client.query(preview_sql).to_dataframe()
df_preview

,Applicant_ID,Age,Employment_Status,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,Device_Type,Application_Date,Fraudulent
0,514,30,Unemployed,55485,3,2399,False,NaT,False,1,172.12.247.187,Tablet,2024-08-21,0
1,553,27,Self-Employed,0,3,3104,False,NaT,False,1,202.18.34.83,Desktop,2024-11-04,0
2,1098,37,Employed,17427,2,2449,False,NaT,True,1,47.79.171.170,Desktop,2024-05-19,0
3,1164,35,Unemployed,0,2,7177,False,NaT,False,1,202.173.250.176,Tablet,2024-10-20,0
4,1991,43,Unemployed,0,1,2876,False,NaT,False,1,72.207.150.138,Tablet,2024-10-28,0
5,2373,27,Unemployed,0,1,6573,False,NaT,True,1,72.207.220.47,Mobile,2024-10-27,0
6,3708,29,Unemployed,33378,1,3343,False,NaT,False,1,207.235.33.85,Mobile,2024-08-23,0
7,3727,49,Unemployed,43791,1,3563,False,NaT,False,1,128.111.58.137,Mobile,2024-07-01,0
8,4172,27,Employed,56567,1,1853,False,NaT,True,1,22.146.189.64,Tablet,2024-07-26,0
9,4238,59,Self-Employed,0,1,2412,False,NaT,False,1,140.20.138.185,Mobile,2024-02-16,0


In [23]:
# --- Helpers ---
def sanitize_col_suffix(val: str) -> str:
    # safe column suffix for one-hot columns
    s = val.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"(^_+|_+$)", "", s)
    return s[:80] if s else "unknown"

# Pull schema from INFORMATION_SCHEMA
schema_sql = f"""
SELECT column_name, data_type
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = '{RAW_TABLE}'
ORDER BY ordinal_position
"""
schema_df = client.query(schema_sql).to_dataframe()
cols = schema_df["column_name"].tolist()
types = dict(zip(schema_df["column_name"], schema_df["data_type"]))

print("Columns detected:", cols)

# Identify boolean columns (native BOOL) + also detect common string bool columns
bool_cols = [c for c in cols if types.get(c, "").upper() in ("BOOL", "BOOLEAN")]

# Heuristic: string columns that look like boolean flags (optional)
string_bool_candidates = []
for c in cols:
    if types.get(c, "").upper() in ("STRING",):
        if re.search(r"(flag|is_|has_|fraud|true|false)", c.lower()):
            string_bool_candidates.append(c)

print("Native BOOL columns:", bool_cols)
print("STRING bool candidates:", string_bool_candidates)

# Detect likely entity ID column for "previous assistance" calc
id_candidates = [c for c in cols if re.search(r"(customer|applicant|user|client).*id|_id$", c.lower())]
id_col = id_candidates[0] if id_candidates else None

# Detect likely timestamp/date column for ordering
time_candidates = [c for c in cols if types.get(c, "").upper() in ("TIMESTAMP","DATETIME","DATE")]
# prefer columns with these keywords
preferred_time = [c for c in time_candidates if re.search(r"(assist|application|request|time|date|ts)", c.lower())]
time_col = (preferred_time[0] if preferred_time else (time_candidates[0] if time_candidates else None))

print("Chosen id_col:", id_col)
print("Chosen time_col:", time_col)

# Fetch distinct values for one-hot encoding for the two categorical fields (if they exist)
def distinct_values(colname):
    if colname not in cols:
        return []
    q = f"SELECT DISTINCT CAST({colname} AS STRING) AS v FROM `{raw_table_id}` WHERE {colname} IS NOT NULL"
    return [r["v"] for r in client.query(q).result()]

emp_vals = distinct_values("Employment_Status")
dev_vals = distinct_values("Device_Type")

print("Employment_Status distinct values:", emp_vals[:20], "..." if len(emp_vals)>20 else "")
print("Device_Type distinct values:", dev_vals[:20], "..." if len(dev_vals)>20 else "")

# Build one-hot expressions
emp_onehots = []
for v in emp_vals:
    suffix = sanitize_col_suffix(v)
    emp_onehots.append(f"IF(CAST(Employment_Status AS STRING) = '{v}', 1, 0) AS emp_{suffix}")

dev_onehots = []
for v in dev_vals:
    suffix = sanitize_col_suffix(v)
    dev_onehots.append(f"IF(CAST(Device_Type AS STRING) = '{v}', 1, 0) AS device_{suffix}")

# Age bins (adjust bins if you want; these are common)
age_bins = [
    ("18_24", "Age BETWEEN 18 AND 24"),
    ("25_34", "Age BETWEEN 25 AND 34"),
    ("35_44", "Age BETWEEN 35 AND 44"),
    ("45_54", "Age BETWEEN 45 AND 54"),
    ("55_64", "Age BETWEEN 55 AND 64"),
    ("65_plus", "Age >= 65"),
]
age_onehots = []
if "Age" in cols:
    for label, cond in age_bins:
        age_onehots.append(f"IF({cond}, 1, 0) AS age_{label}")

# Build base column select list: keep all raw columns EXCEPT the ones we will encode/replace
drop_raw = set([c for c in ["Employment_Status","Device_Type","Age"] if c in cols])

# For booleans: replace column with INT64 equivalent (same name)
# If it's BOOL/BOOLEAN: CAST(col AS INT64)
# If it's string bool candidate: map TRUE/FALSE strings to 1/0
select_exprs = []
for c in cols:
    if c in drop_raw:
        continue

    if c in bool_cols:
        select_exprs.append(f"CAST({c} AS INT64) AS {c}")
    elif c in string_bool_candidates:
        select_exprs.append(f"""
        CASE
          WHEN LOWER(CAST({c} AS STRING)) IN ('true','t','1','yes','y') THEN 1
          WHEN LOWER(CAST({c} AS STRING)) IN ('false','f','0','no','n') THEN 0
          ELSE NULL
        END AS {c}""".strip())
    else:
        select_exprs.append(c)

# Ratio feature (requires columns Income and Amount_Requested)
ratio_expr = None
if "Income" in cols and "Amount_Requested" in cols:
    ratio_expr = "SAFE_DIVIDE(CAST(Income AS FLOAT64), NULLIF(CAST(Amount_Requested AS FLOAT64), 0.0)) AS Income_to_Amount_Requested"

# Time since previous assistance feature
# If we can't detect id/time columns, we still create the field as NULL to satisfy schema expectation.
time_since_expr = None
if id_col and time_col:
    # Handle DATE vs TIMESTAMP types
    if types[time_col].upper() == "DATE":
        time_since_expr = f"DATE_DIFF({time_col}, prev_{time_col}, DAY) AS Time_Since_Previous_Assistance"
    else:
        # TIMESTAMP or DATETIME
        time_since_expr = f"TIMESTAMP_DIFF(TIMESTAMP({time_col}), TIMESTAMP(prev_{time_col}), DAY) AS Time_Since_Previous_Assistance"
else:
    time_since_expr = "CAST(NULL AS INT64) AS Time_Since_Previous_Assistance"

# Compose final SQL
with_base = ""
if id_col and time_col:
    with_base = f"""
WITH base AS (
  SELECT
    *,
    LAG({time_col}) OVER (PARTITION BY {id_col} ORDER BY {time_col}) AS prev_{time_col}
  FROM `{raw_table_id}`
)
"""
    from_clause = "FROM base"
else:
    from_clause = f"FROM `{raw_table_id}`"

engineered_exprs = []
if ratio_expr:
    engineered_exprs.append(ratio_expr)
engineered_exprs.append(time_since_expr)
engineered_exprs.extend(emp_onehots)
engineered_exprs.extend(dev_onehots)
engineered_exprs.extend(age_onehots)

final_sql = f"""
CREATE OR REPLACE TABLE `{train_table_id}` AS
{with_base}
SELECT
  {",\n  ".join(select_exprs)}
  {( "," if engineered_exprs else "")}
  {",\n  ".join(engineered_exprs)}
{from_clause}
"""

print("----- Generated SQL (first 1500 chars) -----")
print(final_sql[:1500])
print("----- end preview -----")

Columns detected: ['Applicant_ID', 'Age', 'Employment_Status', 'Income', 'Number_of_Dependents', 'Amount_Requested', 'Previous_Assistance_Received', 'Previous_Assistance_Date', 'Supporting_Doc_Verified', 'Application_Frequency_Last_Year', 'IP_Address', 'Device_Type', 'Application_Date', 'Fraudulent']
Native BOOL columns: ['Previous_Assistance_Received', 'Supporting_Doc_Verified']
STRING bool candidates: []
Chosen id_col: Applicant_ID
Chosen time_col: Previous_Assistance_Date
Employment_Status distinct values: ['Unemployed', 'Self-Employed', 'Employed'] 
Device_Type distinct values: ['Tablet', 'Desktop', 'Mobile'] 
----- Generated SQL (first 1500 chars) -----

CREATE OR REPLACE TABLE `qwiklabs-gcp-02-5c5074b12ae4.fraud_lab.fraud_training_data` AS

WITH base AS (
  SELECT
    *,
    LAG(Previous_Assistance_Date) OVER (PARTITION BY Applicant_ID ORDER BY Previous_Assistance_Date) AS prev_Previous_Assistance_Date
  FROM `qwiklabs-gcp-02-5c5074b12ae4.fraud_lab.fraud_data_raw`
)

SELECT
  App

In [27]:
job = client.query(final_sql, location=BQ_LOCATION)
job.result()
print("Created:", train_table_id)

# Validate
check_sql = f"SELECT * FROM `{train_table_id}` LIMIT 10"
df_train_preview = client.query(check_sql).to_dataframe()
df_train_preview

Created: qwiklabs-gcp-02-5c5074b12ae4.fraud_lab.fraud_training_data


,Applicant_ID,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,Application_Date,...,emp_employed,device_tablet,device_desktop,device_mobile,age_18_24,age_25_34,age_35_44,age_45_54,age_55_64,age_65_plus
0,5633,0,0,4218,0,NaT,0,1,21.192.43.102,2024-11-29,...,1,0,0,1,0,0,0,1,0,0
1,930,17083,2,3477,0,NaT,0,1,66.204.55.147,2024-06-23,...,0,1,0,0,0,1,0,0,0,0
2,36550,0,2,1850,0,NaT,1,1,245.225.87.164,2024-09-04,...,0,1,0,0,0,1,0,0,0,0
3,18814,0,4,2610,0,NaT,1,1,134.91.61.35,2024-03-28,...,1,1,0,0,0,0,0,0,1,0
4,25816,43950,1,6523,0,NaT,0,1,47.79.91.83,2024-02-11,...,0,0,1,0,0,0,0,0,1,0
5,32510,45500,0,3186,0,NaT,0,1,213.103.155.242,2024-07-14,...,1,0,0,1,0,1,0,0,0,0
6,42401,0,3,9296,0,NaT,1,1,202.173.70.70,2024-01-26,...,0,0,1,0,1,0,0,0,0,0
7,40834,0,4,5588,0,NaT,1,1,129.60.181.130,2024-03-07,...,0,0,1,0,0,0,0,1,0,0
8,27518,21756,5,2692,0,NaT,1,1,129.60.154.255,2024-08-30,...,1,0,1,0,0,0,0,1,0,0
9,4619,34054,2,1792,0,NaT,0,1,202.18.5.152,2024-06-18,...,1,0,1,0,0,0,0,1,0,0


In [25]:
meta_sql = f"""
SELECT column_name, data_type
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = '{TRAIN_TABLE}'
ORDER BY ordinal_position
"""
client.query(meta_sql).to_dataframe()

,column_name,data_type
0,Applicant_ID,INT64
1,Income,INT64
2,Number_of_Dependents,INT64
3,Amount_Requested,INT64
4,Previous_Assistance_Received,INT64
5,Previous_Assistance_Date,DATE
6,Supporting_Doc_Verified,INT64
7,Application_Frequency_Last_Year,INT64
8,IP_Address,STRING
9,Application_Date,DATE


In [30]:
!git config --global user.name "YOUR NAME"
!git config --global user.email "YOUR EMAIL"

# if your notebook folder is not a git repo yet:
!git init
!git add .
!git commit -m "Challenge 1: BigQuery feature engineering for fraud detection"

# create a new repo in GitHub, then add its remote URL:
!git remote add origin https://github.com/<your-username>/<your-repo>.git
!git branch -M main
!git push -u origin main

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
[master (root-commit) 75001ba] Challenge 1: BigQuery feature engineering for fraud detection
 6 files changed, 2 insertions(+)
 create mode 100644 .config/access_tokens.db
 create mode 100644 .config/active_config
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/credentials.db
 create mode 100644 .config/default_configs.db
 create mode 100644 .config/gce
/bin/bash: line 1: your-username: No such file or directory
fatal: 'or